In [12]:
# Laod Env Variables    
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [13]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [14]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

    

In [15]:
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)



In [16]:
# The run_prompt Function
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [17]:
# The run_test_case Function
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [18]:
# The run_eval Function

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [19]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [20]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Regular Expression\n\nHere's a comprehensive regex solution:\n\n```regex\n^(?!-)[a-z0-9][a-z0-9-]{1,61}[a-z0-9]$|^[a-z0-9]{3}$\n```\n\n## Explanation\n\nLet me break down the regex into two parts:\n\n### Part 1: `^(?!-)[a-z0-9][a-z0-9-]{1,61}[a-z0-9]$`\n- `^` - Start of string\n- `(?!-)` - Negative lookahead: ensure it doesn't start with hyphen\n- `[a-z0-9]` - First character must be lowercase letter or number\n- `[a-z0-9-]{1,61}` - Middle section: 1-61 characters of letters, numbers, or hyphens\n- `[a-z0-9]` - Last character must be lowercase letter or number\n- `$` - End of string\n- **Total length**: 3-63 characters \u2713\n\n### Part 2: `|^[a-z0-9]{3}$`\n- Handles the edge case of exactly 3-character names (no middle section needed)\n\n## Key Features\n\n\u2705 **3-63 characters** - Enforced by structure  \n\u2705 **Lowercase only** - `[a-z0-9]` (no uppercase)  \n\u2705 **No leading/trailing hyphens** - First and last chars are `[a-z0-9]`  